# 13 — Cross-encoder reranking

> **Run order.** This notebook is step 13 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.

Everything so far embeds the question and the passage **separately** and compares
the two vectors — a bi-encoder. It never sees the pair side by side.

A **cross-encoder** reads the question and passage *together* and scores the
pair. It is far more accurate and far too slow to run over 9,982 chunks, so it
only ever reorders a shortlist.

**Which makes recall at the shortlist depth a hard ceiling.** This is why the
reranker was deliberately left until last: on the Day 3 baseline the ceiling at
depth 100 was 0.273, so a perfect reranker could not have exceeded that. Query
expansion moved it to 0.614, and that is what makes this step worth doing now.

---

## Re-measured after ADR-008

The verdict below was reached on the ADR-007 index, where **recall@100 was
0.477** - a hard ceiling on anything that only reorders. [ADR-008](../docs/adr/0008-contextual-chunk-prefixes.md)
moved that to **0.841**, so the reason the reranker was rejected no longer
holds and the decision has to be re-tested rather than assumed.


In [1]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.embedding import RERANK_MODEL, Reranker
from analyst.retrievers import dense, hybrid, open_hybrid, open_store, reranked

MODEL = "bge-small"
DEPTH = 100   # shortlist size; recall@DEPTH of the inner retriever caps this
VARIANT = "ctx"   # "" = the ADR-007 index, "ctx" = ADR-008's

settings = get_settings()
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
embedder, store = open_store(settings, MODEL, VARIANT)
_, sparse, hstore = open_hybrid(settings, MODEL, VARIANT)
reranker = Reranker()
print(f"reranker: {RERANK_MODEL}   shortlist depth: {DEPTH}")
print(f"index   : {store.collection} ({store.count():,} pts) / "
      f"{hstore.collection} ({hstore.count():,} pts)")

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


reranker: Xenova/ms-marco-MiniLM-L-6-v2   shortlist depth: 100

index   : elements_ctx_bge-small (10,181 pts) / elements_hybrid_ctx_bge-small (10,181 pts)

## Score both expanded retrievers, reranked

The reranker composes over any `SearchFn`, so this is the same wrapper applied
twice — no retrieval code changes.

In [2]:
suffix = f"[{VARIANT}]" if VARIANT else ""
inners = {
    f"dense+expand{suffix}": dense(embedder, store, "ticker+year", expand=True),
    f"hybrid+expand{suffix}": hybrid(embedder, sparse, hstore, "ticker+year", expand=True),
}

# ~5.3 s per query on this CPU, so ~4 minutes per configuration. The un-reranked
# rows are already in the ledger from notebook 14 - not recomputed here.
for name, inner in inners.items():
    label = f"{name}+rerank"
    search = reranked(inner, reranker, depth=DEPTH, expand=True)
    run = ev.build_run(
        ev.RunConfig(retriever=label, model=MODEL, filters="ticker+year",
                     limit=max(ev.K_VALUES), points=store.count(),
                     notes=f"reranked over the {VARIANT or 'adr-007'} index"),
        ev.evaluate(questions, search, limit=max(ev.K_VALUES)),
        questions,
        root=ev.ROOT,
    )
    ev.append_run(run)
    m = run.metrics
    print(f"{label:<30} R@1 {m.recall_at[1]:.3f}  R@5 {m.recall_at[5]:.3f}  "
          f"R@10 {m.recall_at[10]:.3f}  MRR {m.mrr:.3f}  p50 {m.p50_ms:.0f} ms")

dense+expand[ctx]+rerank       R@1 0.023  R@5 0.204  R@10 0.341  MRR 0.097  p50 4029 ms

hybrid+expand[ctx]+rerank      R@1 0.023  R@5 0.114  R@10 0.182  MRR 0.061  p50 3814 ms

## Did reranking help?

Compared against each retriever's own un-reranked run, so the delta isolates the
reranker rather than mixing in the expansion gain.

In [3]:
ledger = ev.load_runs()
latest = {}
for r in ledger:
    if r.config.model == MODEL and r.config.filters == "ticker+year":
        latest[r.config.retriever] = r

order = ["dense+expand", "dense+expand+rerank",
         "hybrid+expand", "hybrid+expand+rerank",
         "dense+expand[ctx]", "dense+expand[ctx]+rerank",
         "hybrid+expand[ctx]", "hybrid+expand[ctx]+rerank"]
table = pd.DataFrame([latest[k].row() for k in order if k in latest])
print(table.to_string(index=False))

print("\nwhat reranking is worth, on each index:")
for base_name in ("dense+expand", "hybrid+expand", "dense+expand[ctx]", "hybrid+expand[ctx]"):
    rr = f"{base_name}+rerank"
    if base_name in latest and rr in latest:
        b, a = latest[base_name].metrics, latest[rr].metrics
        print(f"  {base_name:<22} R@5 {b.recall_at[5]:.3f} -> {a.recall_at[5]:.3f}  "
              f"({a.recall_at[5] - b.recall_at[5]:+.3f})   "
              f"R@10 {b.recall_at[10]:.3f} -> {a.recall_at[10]:.3f}  "
              f"({a.recall_at[10] - b.recall_at[10]:+.3f})   "
              f"MRR {b.mrr:.3f} -> {a.mrr:.3f}  ({a.mrr - b.mrr:+.3f})")

                                         run                 retriever     model     filters    R@1    R@3    R@5   R@10    MRR   pR@5  points  p50_ms    bench           git
             dense+expand-bge-small-b60d0b39              dense+expand bge-small ticker+year 0.0455 0.0455 0.0682 0.1136 0.0559 0.1364    9982    87.7 2c4aedf3 2f3c9a3-dirty
      dense+expand+rerank-bge-small-b261fe8f       dense+expand+rerank bge-small ticker+year 0.0227 0.0682 0.0909 0.0909 0.0462 0.1364    9982  5313.6 2c4aedf3 2f3c9a3-dirty
            hybrid+expand-bge-small-6510b044             hybrid+expand bge-small ticker+year 0.0455 0.0455 0.0682 0.0909 0.0544 0.1136    9982    91.1 2c4aedf3 2f3c9a3-dirty
     hybrid+expand+rerank-bge-small-51aef45a      hybrid+expand+rerank bge-small ticker+year 0.0227 0.0682 0.0909 0.0909 0.0462 0.1364    9982  5380.9 2c4aedf3 2f3c9a3-dirty
        dense+expand[ctx]-bge-small-f1b65882         dense+expand[ctx] bge-small ticker+year 0.1818 0.2273 0.3182 0.4545 0.2428 0.


what reranking is worth, on each index:

  dense+expand           R@5 0.068 -> 0.091  (+0.023)   R@10 0.114 -> 0.091  (-0.023)   MRR 0.056 -> 0.046  (-0.010)

  hybrid+expand          R@5 0.068 -> 0.091  (+0.023)   R@10 0.091 -> 0.091  (+0.000)   MRR 0.054 -> 0.046  (-0.008)

  dense+expand[ctx]      R@5 0.318 -> 0.204  (-0.114)   R@10 0.455 -> 0.341  (-0.114)   MRR 0.243 -> 0.097  (-0.146)

  hybrid+expand[ctx]     R@5 0.341 -> 0.114  (-0.227)   R@10 0.409 -> 0.182  (-0.227)   MRR 0.209 -> 0.061  (-0.148)

## The cost

The reranker runs the cross-encoder over `DEPTH` passages per question, so its
latency is the honest price of the accuracy. Compare `p50_ms` above against the
un-reranked rows: this is the number that decides whether it ships.

In [4]:
cost = pd.DataFrame([
    {"retriever": k, "p50_ms": latest[k].metrics.p50_ms, "R@5": latest[k].metrics.recall_at[5]}
    for k in order if k in latest
])
print(cost.to_string(index=False))

                retriever  p50_ms    R@5
             dense+expand    87.7 0.0682
      dense+expand+rerank  5313.6 0.0909
            hybrid+expand    91.1 0.0682
     hybrid+expand+rerank  5380.9 0.0909
        dense+expand[ctx]    91.7 0.3182
 dense+expand[ctx]+rerank  4029.4 0.2045
       hybrid+expand[ctx]    90.0 0.3409
hybrid+expand[ctx]+rerank  3814.2 0.1136

## The ledger

In [5]:
print(ev.write_leaderboard(ev.load_runs()))

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\results\leaderboard.md